In [1]:
# ── 0. Configuración del entorno ──────────────────────────────────────────────
"""
Entorno   : .venv  (Python 3.12)
Deps      : requirements.txt  (generado con pip freeze)
Motor DB  : PostgreSQL 17.8  →  base tesis_db, tabla inclusion_financiera
"""

# ── Librerías ─────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from sqlalchemy import create_engine, text
from pathlib import Path

# ── Semilla global (reproducibilidad en métodos estocásticos) ─────────────────
SEED = 42
np.random.seed(SEED)

# ── Estilo de gráficos ───────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", context="notebook", font_scale=1.1)
plt.rcParams.update({
    "figure.figsize": (10, 5),
    "figure.dpi": 120,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "savefig.bbox": "tight",
    "savefig.dpi": 200,
})
PALETTE = sns.color_palette("Set2")

# ── Opciones de pandas ────────────────────────────────────────────────────────
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

# ── Rutas del proyecto ────────────────────────────────────────────────────────
ROOT = Path.cwd()
DATA_DIR = ROOT          # Archivos .xlsx viven aquí
SRC_DIR  = ROOT / "src"  # Módulo con funciones reutilizables

print("✓ Configuración cargada")

✓ Configuración cargada


In [ ]:
# ── 1. Conexión a PostgreSQL ──────────────────────────────────────────────────
# Creamos un engine de SQLAlchemy que actúa como "puente" entre Python y la base
# de datos PostgreSQL. Con este engine podemos ejecutar queries SQL y cargar
# resultados directamente a DataFrames de pandas.
# Como verificación, consultamos el número de filas y columnas de la tabla
# para confirmar que la conexión funciona y la tabla existe.

DB_URL = "postgresql://anapaulaperezgavilan@localhost:5432/tesis_db"
engine = create_engine(DB_URL)

with engine.connect() as conn:
    # Conteo de registros en la tabla principal
    n_rows = conn.execute(text("SELECT COUNT(*) FROM inclusion_financiera")).scalar()
    # Conteo de columnas consultando el catálogo de PostgreSQL (information_schema)
    n_cols = conn.execute(text(
        "SELECT COUNT(*) FROM information_schema.columns "
        "WHERE table_name = 'inclusion_financiera'"
    )).scalar()

print(f"✓ Conectado a tesis_db")
print(f"  Tabla inclusion_financiera: {n_rows:,} registros × {n_cols} columnas")

✓ Conectado a tesis_db
  Tabla inclusion_financiera: 41,905 registros × 175 columnas


## 2. Carga de datos
Lectura completa de la tabla `inclusion_financiera` a un DataFrame.  
Verificamos dimensiones y tipos.

In [ ]:
# ── 2a. Cargar tabla completa ─────────────────────────────────────────────────
# Leemos TODA la tabla inclusion_financiera a un DataFrame de pandas.
# Esto nos permite trabajar con los datos en memoria sin hacer queries repetidos.
# Verificamos: dimensiones (filas × columnas), uso de memoria, y distribución
# de tipos de datos (int64, float64, object) para confirmar que la carga fue correcta.

df = pd.read_sql("SELECT * FROM inclusion_financiera", engine)

print(f"Dimensiones: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"Memoria: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"\nTipos de columna:")
print(df.dtypes.value_counts().to_string())
df.head(3)

Dimensiones: 41,905 filas × 175 columnas
Memoria: 74.8 MB

Tipos de columna:
int64      121
float64     46
object       8


,cve_mun,trim,cve_edo,region,estado,municipio,pob,pob_adulta,pob_adulta_m,pob_adulta_h,tipo_pob,ncont_ahorro_m,ncont_ahorro_h,ncont_ahorro_pm,ncont_ahorro_t,ncont_plazo_m,ncont_plazo_h,ncont_plazo_pm,ncont_plazo_t,ncont_n1_m,ncont_n1_h,ncont_n1_pm,ncont_n1_t,ncont_n2_m,ncont_n2_h,...,flag_undef_saldoprom_ahorro_t,flag_undef_saldoprom_plazo_m,flag_undef_saldoprom_plazo_h,flag_undef_saldoprom_plazo_pm,flag_undef_saldoprom_plazo_t,flag_undef_saldoprom_n1_m,flag_undef_saldoprom_n1_h,flag_undef_saldoprom_n1_pm,flag_undef_saldoprom_n1_t,flag_undef_saldoprom_n2_m,flag_undef_saldoprom_n2_h,flag_undef_saldoprom_n2_pm,flag_undef_saldoprom_n2_t,flag_undef_saldoprom_n3_m,flag_undef_saldoprom_n3_h,flag_undef_saldoprom_n3_pm,flag_undef_saldoprom_n3_t,flag_undef_saldoprom_tradic_m,flag_undef_saldoprom_tradic_h,flag_undef_saldoprom_tradic_pm,flag_undef_saldoprom_tradic_t,flag_undef_saldoprom_total_m,flag_undef_saldoprom_total_h,flag_undef_saldoprom_total_pm,flag_undef_saldoprom_total_t
0,20223,221,20,Sur,Oaxaca,San Juan Yatzona,440,312,178,134,Rural,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
1,20142,318,20,Sur,Oaxaca,San Francisco Huehuetlan,1053,827,467,360,Rural,0,0,0,0,0,0,0,0,0,0,0,0,3,2,...,1,1,1,1,1,1,1,1,1,0,0,1,0,1,1,1,1,1,1,1,1,0,0,1,0
2,20196,318,20,Sur,Oaxaca,San Juan Evangelista Analco,381,296,154,141,Rural,0,0,0,0,0,0,0,0,0,0,0,0,5,1,...,1,1,1,1,1,1,1,1,1,0,0,1,0,1,1,1,1,1,1,1,1,0,0,1,0


In [ ]:
# ── 2b. Verificación de llave primaria ────────────────────────────────────────
# La llave primaria del panel es (cve_mun, periodo_trimestre).
# Cada combinación municipio-trimestre debe aparecer exactamente UNA vez.
# Si el assert falla, hay duplicados que debemos investigar antes de continuar.
# También mostramos el primer y último periodo para confirmar el rango temporal.

pk_unique = df.groupby(["cve_mun", "periodo_trimestre"]).size()
assert (pk_unique == 1).all(), "⚠ Hay duplicados en la llave primaria"
print(f"✓ Llave primaria (cve_mun, periodo_trimestre) es única")
print(f"  Municipios: {df['cve_mun'].nunique():,}")
print(f"  Trimestres: {df['periodo_trimestre'].nunique()}")
print(f"  Periodos:   {sorted(df['periodo_trimestre'].unique())[[0, -1]]}")

## 2c. Inspección estructural mínima

In [ ]:
# ── 2c-i. df.info() — Tipos, conteos no-NA y memoria ─────────────────────────
# df.info() muestra para CADA columna:
#   - Su nombre
#   - Cuántos valores no-nulos tiene (Non-Null Count)
#   - Su tipo de dato (dtype)
# Al final reporta el uso total de memoria (deep = incluye el contenido real
# de las columnas object/string, no solo los punteros).
# Esto nos da el primer vistazo de la "salud" de cada columna.

df.info(verbose=True, show_counts=True, memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41905 entries, 0 to 41904
Data columns (total 175 columns):
 #    Column                          Non-Null Count  Dtype  
---   ------                          --------------  -----  
 0    cve_mun                         41905 non-null  int64  
 1    trim                            41905 non-null  int64  
 2    cve_edo                         41905 non-null  int64  
 3    region                          41905 non-null  object 
 4    estado                          41905 non-null  object 
 5    municipio                       41905 non-null  object 
 6    pob                             41905 non-null  int64  
 7    pob_adulta                      41905 non-null  int64  
 8    pob_adulta_m                    41905 non-null  int64  
 9    pob_adulta_h                    41905 non-null  int64  
 10   tipo_pob                        41903 non-null  object 
 11   ncont_ahorro_m                  41905 non-null  int64  
 12   ncont_ahorro_h  

In [ ]:
# ── 2c-ii. df.describe() — Estadística descriptiva (numéricas) ────────────────
# df.describe() calcula para cada columna numérica:
#   count  = valores no-NA (si < 41,905 → hay NULLs)
#   mean   = promedio
#   std    = desviación estándar (si = 0 → columna constante, no aporta variación)
#   min    = valor mínimo (si < 0 en variables que no deberían → posible error)
#   25/50/75%  = cuartiles
#   max    = valor máximo
# Las columnas object (texto) se excluyen automáticamente.

df.describe()

,cve_mun,trim,cve_edo,pob,pob_adulta,pob_adulta_m,pob_adulta_h,ncont_ahorro_m,ncont_ahorro_h,ncont_ahorro_pm,ncont_ahorro_t,ncont_plazo_m,ncont_plazo_h,ncont_plazo_pm,ncont_plazo_t,ncont_n1_m,ncont_n1_h,ncont_n1_pm,ncont_n1_t,ncont_n2_m,ncont_n2_h,ncont_n2_pm,ncont_n2_t,ncont_n3_m,ncont_n3_h,...,flag_undef_saldoprom_ahorro_t,flag_undef_saldoprom_plazo_m,flag_undef_saldoprom_plazo_h,flag_undef_saldoprom_plazo_pm,flag_undef_saldoprom_plazo_t,flag_undef_saldoprom_n1_m,flag_undef_saldoprom_n1_h,flag_undef_saldoprom_n1_pm,flag_undef_saldoprom_n1_t,flag_undef_saldoprom_n2_m,flag_undef_saldoprom_n2_h,flag_undef_saldoprom_n2_pm,flag_undef_saldoprom_n2_t,flag_undef_saldoprom_n3_m,flag_undef_saldoprom_n3_h,flag_undef_saldoprom_n3_pm,flag_undef_saldoprom_n3_t,flag_undef_saldoprom_tradic_m,flag_undef_saldoprom_tradic_h,flag_undef_saldoprom_tradic_pm,flag_undef_saldoprom_tradic_t,flag_undef_saldoprom_total_m,flag_undef_saldoprom_total_h,flag_undef_saldoprom_total_pm,flag_undef_saldoprom_total_t
count,"41,905.00","41,905.00","41,905.00","41,905.00","41,905.00","41,905.00","41,905.00","41,905.00","41,905.00","41,905.00","41,905.00","41,905.00","41,905.00","41,905.00","41,905.00","41,905.00","41,905.00","41,905.00","41,905.00","41,905.00","41,905.00","41,905.00","41,905.00","41,905.00","41,905.00",...,"41,905.00","41,905.00","41,905.00","41,905.00","41,905.00","41,905.00","41,905.00","41,905.00","41,905.00","41,905.00","41,905.00","41,905.00","41,905.00","41,905.00","41,905.00","41,905.00","41,905.00","41,905.00","41,905.00","41,905.00","41,905.00","41,905.00","41,905.00","41,905.00","41,905.00"
mean,"19,338.99",273.06,19.23,"51,115.76","37,884.32","19,702.27","18,182.05",47.54,51.02,0.08,98.64,708.86,664.93,46.03,"1,419.81",4.85,64.35,0.00,69.21,"4,699.65","4,329.37",6.04,"9,035.07",86.60,111.82,...,0.93,0.58,0.58,0.81,0.57,0.97,0.91,1.00,0.91,0.03,0.04,0.85,0.01,0.84,0.83,0.99,0.82,0.57,0.56,0.70,0.56,0.03,0.04,0.70,0.01
std,"7,373.70",108.73,7.36,"146,944.81","111,596.50","58,012.74","53,614.74",401.42,435.66,3.18,835.14,"3,745.19","10,927.47",590.12,"14,021.69",258.34,"4,718.33",0.12,"4,790.74","52,658.32","40,233.55",96.18,"88,466.39",953.72,"1,734.22",...,0.26,0.49,0.49,0.39,0.50,0.17,0.28,0.03,0.29,0.16,0.19,0.36,0.12,0.37,0.38,0.09,0.38,0.50,0.50,0.46,0.50,0.16,0.18,0.46,0.12
min,"1,001.00",119.00,1.00,81.00,66.00,36.00,26.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,"14,080.00",219.00,14.00,"4,552.00","3,290.00","1,710.00","1,585.00",0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,24.00,22.00,0.00,51.00,0.00,0.00,...,1.00,0.00,0.00,1.00,0.00,1.00,1.00,1.00,1.00,0.00,0.00,1.00,0.00,1.00,1.00,1.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
50%,"20,228.00",318.00,20.00,"13,701.00","9,873.00","5,133.00","4,767.00",0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,140.00,136.00,0.00,296.00,0.00,0.00,...,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,0.00,0.00,1.00,0.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,0.00,0.00,1.00,0.00
75%,"24,028.00",322.00,24.00,"35,905.00","25,647.00","13,205.00","12,364.00",0.00,0.00,0.00,0.00,194.00,106.00,0.00,305.00,0.00,0.00,0.00,0.00,"1,022.00",878.00,0.00,"2,060.00",0.00,0.00,...,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,0.00,0.00,1.00,0.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,0.00,0.00,1.00,0.00
max,"32,058.00",421.00,32.00,"1,922,523.00","1,468,256.00","766,847.00","736,875.00","13,130.00","15,965.00",202.00,"28,206.00","138,816.00","594,884.00","26,191.00","718,838.00","22,827.00","445,236.00",11.00,"468,063.00","4,163,526.00","2,387,959.00","11,773.00","5,267,939.00","124,446.00","255,013.00",...,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00


In [ ]:
# ── 2c-iii. df.isna() — Mapa de faltantes ─────────────────────────────────────
# df.isna() genera una máscara booleana (True/False) del mismo tamaño que df,
# donde True = valor faltante (NULL/NaN).
# Sumamos por columna para obtener el CONTEO de NULLs en cada una.
# Filtramos solo las que tienen al menos 1 NULL y las ordenamos de mayor a menor.
# Esto es clave para saber:
#   - ¿Qué columnas están completas? (128 de 175)
#   - ¿Cuáles tienen NULLs y cuántos? (47 columnas)
#   - ¿El patrón de NULLs es por diseño (ej. rezagos, saldoprom) o inesperado?

na_mask = df.isna()
na_summary = na_mask.sum()
na_summary = na_summary[na_summary > 0].sort_values(ascending=False)

print(f"Columnas con al menos 1 NULL: {len(na_summary)} de {df.shape[1]}")
print(f"Columnas 100% completas:      {df.shape[1] - len(na_summary)}")
print(f"Celdas totales:               {df.shape[0] * df.shape[1]:,}")
print(f"Celdas NA:                    {na_mask.sum().sum():,} ({na_mask.sum().sum() / (df.shape[0]*df.shape[1]) * 100:.2f}%)")
print(f"\n{'─'*65}")
print(f"{'Columna':42s} {'NULLs':>8s} {'%':>7s}")
print(f"{'─'*65}")
for col, n in na_summary.items():
    pct = n / df.shape[0] * 100
    print(f"{col:42s} {n:>8,d} {pct:>6.1f}%")

Columnas con al menos 1 NULL: 47 de 175
Columnas 100% completas:      128
Celdas totales:               7,333,375
Celdas NA:                    830,549 (11.33%)

─────────────────────────────────────────────────────────────────
Columna                                       NULLs       %
─────────────────────────────────────────────────────────────────
saldoprom_n1_pm                              41,871   99.9%
saldoprom_ahorro_pm                          41,721   99.6%
saldoprom_n3_pm                              41,569   99.2%
saldoprom_n1_m                               40,586   96.9%
saldoprom_ahorro_m                           39,267   93.7%
saldoprom_ahorro_h                           39,163   93.5%
saldoprom_ahorro_t                           38,839   92.7%
saldoprom_n1_h                               38,279   91.3%
saldoprom_n1_t                               37,981   90.6%
saldoprom_n2_pm                              35,436   84.6%
saldoprom_n3_m                               3

In [ ]:
# ── 2c-iv. Resumen compacto para BRIEF ────────────────────────────────────────
# Esta celda consolida los hallazgos de info(), describe() e isna() en un
# resumen compacto que se puede copiar directamente al archivo BRIEF.md.
# Se extraen las métricas más relevantes para documentación:
#   - Dimensiones y memoria (de info)
#   - Columnas constantes y valores negativos (de describe)
#   - Desglose de NULLs por grupo de variables (de isna)
#   - Resumen de columnas categóricas (object)

import io

# ── INFO: extraemos solo las líneas clave (dimensiones, tipos, memoria) ──
buf = io.StringIO()
df.info(memory_usage="deep", buf=buf)
info_text = buf.getvalue()
for line in info_text.split("\n"):
    if any(kw in line.lower() for kw in ["rangeindex", "columns", "dtypes", "memory"]):
        print(line.strip())

print()

# ── DESCRIBE: buscamos columnas problemáticas ──
desc = df.describe()
print(f"df.describe(): {desc.shape[1]} columnas numéricas")
# Si count < 41905, esa columna tiene NULLs
print(f"  count rango:  {desc.loc['count'].min():.0f} – {desc.loc['count'].max():.0f}")
print(f"  Cols con count < 41905: {(desc.loc['count'] < 41905).sum()}")

# std=0 → columna constante, no aporta variación → excluir de modelos
zero_std = desc.columns[desc.loc["std"] == 0].tolist()
print(f"  Cols con std=0 (constantes): {len(zero_std)} → {zero_std}")

# min<0 → posible error en variables que deberían ser ≥ 0
neg_min = desc.columns[desc.loc["min"] < 0].tolist()
print(f"  Cols con min < 0: {len(neg_min)} → {neg_min}")

print()

# ── ISNA: desglose de NULLs por grupo de variables ──
na_total = df.isna().sum()
na_cols = na_total[na_total > 0].sort_values(ascending=False)
n_cells = df.shape[0] * df.shape[1]
print(f"NULLs totales: {na_total.sum():,} de {n_cells:,} celdas ({na_total.sum()/n_cells*100:.2f}%)")
print(f"Columnas con NULLs: {len(na_cols)} de {df.shape[1]}")
print(f"Columnas 100% completas: {df.shape[1] - len(na_cols)}")

# Agrupamos por patrón de nombre para identificar la causa de los NULLs:
# - saldoprom_*: NULLs estructurales (÷0, no datos faltantes reales)
# - alcaldesa_*: NULLs por diseño (variantes sin llenado manual + rezagos)
# - otras: casos aislados a investigar
saldoprom_na = {c: v for c, v in na_cols.items() if "saldoprom_" in c and not c.startswith("flag")}
alcaldesa_na = {c: v for c, v in na_cols.items() if "alcaldesa" in c}
other_na = {c: v for c, v in na_cols.items() if c not in saldoprom_na and c not in alcaldesa_na}

print(f"\n  Grupo saldoprom_* ({len(saldoprom_na)} cols): {min(saldoprom_na.values()):,} – {max(saldoprom_na.values()):,} NULLs")
print(f"  Grupo alcaldesa_* ({len(alcaldesa_na)} cols): {min(alcaldesa_na.values()):,} – {max(alcaldesa_na.values()):,} NULLs")
if other_na:
    print(f"  Otras ({len(other_na)} cols):")
    for c, v in other_na.items():
        print(f"    {c}: {v:,} ({v/df.shape[0]*100:.1f}%)")

# ── Columnas categóricas: verificamos cardinalidad y ejemplos ──
print(f"\n── Columnas categóricas (object) ──")
for col in df.select_dtypes(include="object").columns:
    print(f"  {col}: {df[col].nunique()} únicos, ejemplo={df[col].iloc[0]}")

RangeIndex: 41905 entries, 0 to 41904
Columns: 175 entries, cve_mun to flag_undef_saldoprom_total_t
dtypes: float64(46), int64(121), object(8)
memory usage: 71.4 MB

df.describe(): 167 columnas numéricas
  count rango:  34 – 41905
  Cols con count < 41905: 46
  Cols con std=0 (constantes): 3 → ['hist_state_available', 'missing_quarters_alcaldesa', 'ok_panel_completo_final']
  Cols con min < 0: 0 → []

NULLs totales: 830,549 de 7,333,375 celdas (11.33%)
Columnas con NULLs: 47 de 175
Columnas 100% completas: 128

  Grupo saldoprom_* (28 cols): 600 – 41,871 NULLs
  Grupo alcaldesa_* (18 cols): 1,781 – 8,185 NULLs
  Otras (1 cols):
    tipo_pob: 2 (0.0%)

── Columnas categóricas (object) ──
  region: 6 únicos, ejemplo=Sur
  estado: 32 únicos, ejemplo=Oaxaca
  municipio: 2424 únicos, ejemplo=San Juan Yatzona
  tipo_pob: 6 únicos, ejemplo=Rural
  periodo_trimestre: 17 únicos, ejemplo=2021Q2
  cve_ent: 32 únicos, ejemplo=20
  cve_mun3: 570 únicos, ejemplo=223
  cvegeo_mun: 2471 únicos, ejempl

## 3. Diccionario de variables
Cargamos el diccionario externo (`Diccionario_y_QC_Base_CNBV_alcaldesa_v2.xlsx`) y construimos un catálogo programático de las columnas de la base.

In [ ]:
# ── 3a. Diccionario desde archivo Excel ───────────────────────────────────────
# Cargamos el archivo de diccionario que documenta las variables de la base.
# El Excel tiene 4 hojas:
#   00_resumen            → métricas generales (filas, columnas, periodo, etc.)
#   01_diccionario        → definición de cada variable (nombre, tipo, NULLs, grupo)
#   02_manual_por_estado  → filas corregidas manualmente por estado
#   03_snim_por_estado    → filas con fuente SNIM por estado
# Leemos cada hoja a un DataFrame y listamos sus columnas para inspección.

dict_path = DATA_DIR / "Diccionario_y_QC_Base_CNBV_alcaldesa_v2.xlsx"
dict_sheets = pd.ExcelFile(dict_path).sheet_names
print(f"Hojas en diccionario: {dict_sheets}")

# Leer todas las hojas en un dict de DataFrames
diccionario = {sheet: pd.read_excel(dict_path, sheet_name=sheet) for sheet in dict_sheets}
for name, sheet_df in diccionario.items():
    print(f"\n── {name} ({sheet_df.shape[0]} filas × {sheet_df.shape[1]} cols) ──")
    print(f"  Columnas: {list(sheet_df.columns)}")

Hojas en diccionario: ['00_resumen', '01_diccionario', '02_manual_por_estado', '03_snim_por_estado']

── 00_resumen (12 filas × 2 cols) ──
  Columnas: ['metric', 'value']

── 01_diccionario (148 filas × 9 cols) ──
  Columnas: ['variable', 'grupo', 'subgrupo', 'tipo', 'missing_n', 'missing_pct', 'ejemplo_1', 'ejemplo_2', 'definicion']

── 02_manual_por_estado (23 filas × 2 cols) ──
  Columnas: ['estado', 'filas_manual']

── 03_snim_por_estado (17 filas × 2 cols) ──
  Columnas: ['estado', 'filas_snim']


In [ ]:
# ── 3b. Catálogo programático de columnas ─────────────────────────────────────
# Generamos un catálogo AUTOMÁTICO a partir del DataFrame, independiente del
# diccionario Excel. Para cada columna calcula:
#   - dtype:    tipo de dato en pandas
#   - nulls:    cantidad de valores faltantes
#   - pct_null: porcentaje de NULLs
#   - n_unique: valores únicos (cardinalidad)
#   - ejemplo:  primer valor no-nulo
#   - min/max:  rango (solo para columnas numéricas)
# Este catálogo sirve para cruzar contra el diccionario Excel y detectar
# discrepancias (tipos incorrectos, NULLs no documentados, columnas nuevas).

def build_catalog(dataframe: pd.DataFrame) -> pd.DataFrame:
    """Genera un catálogo con tipo, NULLs, únicos, min, max para cada columna."""
    records = []
    for col in dataframe.columns:
        s = dataframe[col]
        rec = {
            "columna": col,
            "dtype": str(s.dtype),
            "nulls": s.isna().sum(),
            "pct_null": round(s.isna().mean() * 100, 2),
            "n_unique": s.nunique(),
            "ejemplo": s.dropna().iloc[0] if s.notna().any() else None,
        }
        if pd.api.types.is_numeric_dtype(s):
            rec["min"] = s.min()
            rec["max"] = s.max()
        records.append(rec)
    return pd.DataFrame(records)

catalogo = build_catalog(df)
print(f"Catálogo: {catalogo.shape[0]} variables")
catalogo

## 3c. Actualización del diccionario Excel
Se aplican las 6 correcciones detectadas en la auditoría:
1. Agregar 31 columnas faltantes
2. Eliminar 4 variables fantasma
3. Corregir 61 tipos
4. Corregir 28 conteos de NULLs
5. Actualizar hoja `00_resumen`
6. Guardar como **v3**

In [10]:
# ── 3c. Actualizar diccionario Excel ──────────────────────────────────────────
# La auditoría del diccionario (v2) contra la base real detectó 6 problemas:
#   1) 4 variables "fantasma" que ya no existen en la base → las eliminamos
#   2) 61 tipos de datos incorrectos (reflejaban estado pre-tipificación) → los corregimos
#   3) 28 conteos de NULLs incorrectos en saldoprom_* → los actualizamos
#   4) 31 columnas nuevas sin documentar (3 IDs INEGI + 28 flags) → las agregamos
#   5) Hoja 00_resumen con conteo desactualizado (148→175) → la actualizamos
# Todo se aplica sobre una copia del diccionario original.

import copy

# Trabajamos sobre una copia del diccionario actual
dict_updated = dict_df.copy()

# ═══════════════════════════════════════════════════════════════════════════════
# 1) ELIMINAR 4 variables fantasma (ya no existen en la base)
#    Estas columnas se eliminaron al migrar de CSV a PostgreSQL, pero el
#    diccionario las seguía listando.
# ═══════════════════════════════════════════════════════════════════════════════
phantom = ["alcaldesa_manual", "fuente_manual", "pm_nombre_manual", "reason_detailed"]
dict_updated = dict_updated[~dict_updated["variable"].isin(phantom)].copy()
print(f"✓ Eliminadas {len(phantom)} variables fantasma: {phantom}")

# ═══════════════════════════════════════════════════════════════════════════════
# 2) CORREGIR TIPOS para las columnas que ya existen
#    El diccionario v2 decía float64 para columnas que se convirtieron a int64
#    en PostgreSQL (ej. pob, saldocont_*), y decía object para saldoprom_*
#    que ahora son float64. Actualizamos al tipo real de pandas.
# ═══════════════════════════════════════════════════════════════════════════════
dtype_map = {"int64": "int64", "float64": "float64", "object": "object"}

updated_types = 0
for idx, row in dict_updated.iterrows():
    var = row["variable"]
    if var in db_vars:
        real_type = dtype_map.get(str(df[var].dtype), str(df[var].dtype))
        if str(row["tipo"]).strip() != real_type:
            dict_updated.at[idx, "tipo"] = real_type
            updated_types += 1
print(f"✓ Corregidos {updated_types} tipos de datos")

# ═══════════════════════════════════════════════════════════════════════════════
# 3) CORREGIR CONTEOS DE NULLs
#    Las 28 columnas saldoprom_* decían missing_n=0 porque cuando eran tipo
#    object, los "-" se contaban como valores presentes. Ahora que son float64,
#    los "-" son NULL y los conteos cambiaron.
# ═══════════════════════════════════════════════════════════════════════════════
updated_nulls = 0
for idx, row in dict_updated.iterrows():
    var = row["variable"]
    if var in db_vars:
        real_nulls = int(df[var].isna().sum())
        real_pct = round(df[var].isna().mean() * 100, 2)
        if pd.notna(row["missing_n"]) and int(row["missing_n"]) != real_nulls:
            dict_updated.at[idx, "missing_n"] = real_nulls
            dict_updated.at[idx, "missing_pct"] = real_pct
            updated_nulls += 1
print(f"✓ Corregidos {updated_nulls} conteos de NULLs")

# ═══════════════════════════════════════════════════════════════════════════════
# 4) AGREGAR 31 columnas faltantes
#    3 identificadores INEGI (cve_ent, cve_mun3, cvegeo_mun) creados el 22/02
#    + 28 flags flag_undef_saldoprom_* creados el mismo día.
#    Para cada una generamos: tipo, NULLs, ejemplos, grupo y definición.
# ═══════════════════════════════════════════════════════════════════════════════
new_rows = []
for col in en_base_no_dict:
    s = df[col]
    new_row = {
        "variable": col,
        "tipo": dtype_map.get(str(s.dtype), str(s.dtype)),
        "missing_n": int(s.isna().sum()),
        "missing_pct": round(s.isna().mean() * 100, 2),
        "ejemplo_1": s.dropna().iloc[0] if s.notna().any() else None,
        "ejemplo_2": s.dropna().iloc[1] if s.notna().sum() > 1 else None,
    }
    # Asignamos grupo y definición según el patrón del nombre
    if col.startswith("flag_undef_"):
        new_row["grupo"] = "Flags missingness"
        new_row["subgrupo"] = "saldoprom_undefined"
        producto_sexo = col.replace("flag_undef_saldoprom_", "")
        new_row["definicion"] = (
            f"1 si saldoprom_{producto_sexo} es NULL por indefinición estructural "
            f"(ncont=0 → ÷0). 0 si valor legítimo."
        )
    elif col == "cve_ent":
        new_row["grupo"] = "Identificadores"
        new_row["subgrupo"] = "claves_INEGI"
        new_row["definicion"] = "Clave de entidad AGEE zero-padded (2 chars, ej. '01')"
    elif col == "cve_mun3":
        new_row["grupo"] = "Identificadores"
        new_row["subgrupo"] = "claves_INEGI"
        new_row["definicion"] = "Clave municipal AGEM zero-padded (3 chars, ej. '001')"
    elif col == "cvegeo_mun":
        new_row["grupo"] = "Identificadores"
        new_row["subgrupo"] = "claves_INEGI"
        new_row["definicion"] = "Clave geoestadística canónica INEGI = cve_ent || cve_mun3 (5 chars, ej. '01001')"
    new_rows.append(new_row)

new_df = pd.DataFrame(new_rows)
dict_updated = pd.concat([dict_updated, new_df], ignore_index=True)
print(f"✓ Agregadas {len(new_rows)} columnas nuevas")

# ═══════════════════════════════════════════════════════════════════════════════
# 5) ACTUALIZAR hoja 00_resumen
#    Cambiamos el conteo de columnas de 148 a 175 para reflejar la realidad.
# ═══════════════════════════════════════════════════════════════════════════════
resumen_updated = resumen_df.copy()
mask = resumen_updated["metric"] == "Columnas"
resumen_updated.loc[mask, "value"] = 175

print(f"\n── Diccionario actualizado: {dict_updated.shape[0]} variables ──")
print(f"   Tipos: {dict_updated['tipo'].value_counts().to_dict()}")

✓ Eliminadas 4 variables fantasma: ['alcaldesa_manual', 'fuente_manual', 'pm_nombre_manual', 'reason_detailed']
✓ Corregidos 61 tipos de datos
✓ Corregidos 28 conteos de NULLs
✓ Agregadas 31 columnas nuevas

── Diccionario actualizado: 175 variables ──
   Tipos: {'int64': 121, 'float64': 46, 'object': 8}


In [11]:
# ── 3d. Guardar diccionario actualizado como v3 ──────────────────────────────
# Exportamos el diccionario corregido a un nuevo archivo Excel (v3) para no
# sobreescribir el original (v2). Incluimos las 4 hojas:
#   00_resumen           → actualizada (175 cols)
#   01_diccionario       → corregida (175 vars, tipos y NULLs actualizados)
#   02_manual_por_estado → sin cambios (se copia del original)
#   03_snim_por_estado   → sin cambios (se copia del original)
# Al final, hacemos una verificación cruzada: el diccionario v3 debe tener
# exactamente las mismas 175 variables que la base de datos. Si quedan
# diferencias, hay que investigar.

out_path = DATA_DIR / "Diccionario_y_QC_Base_CNBV_alcaldesa_v3.xlsx"

with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    resumen_updated.to_excel(writer, sheet_name="00_resumen", index=False)
    dict_updated.to_excel(writer, sheet_name="01_diccionario", index=False)
    diccionario["02_manual_por_estado"].to_excel(writer, sheet_name="02_manual_por_estado", index=False)
    diccionario["03_snim_por_estado"].to_excel(writer, sheet_name="03_snim_por_estado", index=False)

print(f"✓ Diccionario guardado: {out_path.name}")
print(f"  Hojas: 00_resumen, 01_diccionario, 02_manual_por_estado, 03_snim_por_estado")

# ── Verificación cruzada final ────────────────────────────────────────────────
# Releemos el v3 y comparamos contra las columnas reales de la base.
# Esperamos 0 diferencias en ambas direcciones.
v3 = pd.read_excel(out_path, sheet_name="01_diccionario")
v3_vars = set(v3["variable"].dropna().str.strip())

still_missing = sorted(db_vars - v3_vars)
still_phantom = sorted(v3_vars - db_vars)
print(f"\n── Verificación cruzada v3 vs DB ──")
print(f"  Variables en v3:              {len(v3_vars)}")
print(f"  Columnas en DB:               {len(db_vars)}")
print(f"  En DB pero no en v3:          {len(still_missing)} → {still_missing}")
print(f"  En v3 pero no en DB:          {len(still_phantom)} → {still_phantom}")

✓ Diccionario guardado: Diccionario_y_QC_Base_CNBV_alcaldesa_v3.xlsx
  Hojas: 00_resumen, 01_diccionario, 02_manual_por_estado, 03_snim_por_estado

── Verificación cruzada v3 vs DB ──
  Variables en v3:              175
  Columnas en DB:               175
  En DB pero no en v3:          0 → []
  En v3 pero no en DB:          0 → []


## 4. Consultas SQL directas
Puedes ejecutar queries SQL sin cargar todo el DataFrame:

In [ ]:
# ── Ejemplo 1: Proporción de alcaldesas por estado y año ──────────────────────
# Calculamos el promedio de alcaldesa_final (0/1) por estado y año.
# Como es binaria, su promedio = proporción de municipios-trimestre con alcaldesa.
# Esto nos da una vista de la distribución del "tratamiento" a lo largo del
# tiempo y la geografía — fundamental para evaluar si hay suficiente variación.

query = """
SELECT estado, year, 
       ROUND(AVG(alcaldesa_final)::numeric, 3) AS prop_alcaldesa,
       COUNT(*) AS n_municipios
FROM inclusion_financiera
GROUP BY estado, year
ORDER BY estado, year
"""
pd.read_sql(query, engine)

,estado,year,prop_alcaldesa,n_municipios
0,Aguascalientes,2018,0.273,22
1,Aguascalientes,2019,0.250,44
2,Aguascalientes,2020,0.182,44
3,Aguascalientes,2021,0.182,44
4,Aguascalientes,2022,0.182,33
...,...,...,...,...
155,Zacatecas,2018,0.250,116
156,Zacatecas,2019,0.224,232
157,Zacatecas,2020,0.224,232
158,Zacatecas,2021,0.220,232


In [ ]:
# ── Ejemplo 2: Inclusión financiera promedio por tipo de alcaldesa ─────────────
# Comparación descriptiva (NO causal) de indicadores de inclusión financiera
# entre municipios-trimestre con alcaldesa (=1) vs sin alcaldesa (=0).
# Esto muestra las diferencias brutas — para inferencia causal se necesitan
# controles, efectos fijos y diseño econométrico adecuado.

query = """
SELECT 
    alcaldesa_final,
    ROUND(AVG(ncont_total_t)::numeric, 0) AS avg_contratos,
    ROUND(AVG(numtar_deb_t)::numeric, 0) AS avg_tarjetas_debito,
    ROUND(AVG(numtar_cred_t)::numeric, 0) AS avg_tarjetas_credito,
    COUNT(*) AS obs
FROM inclusion_financiera
GROUP BY alcaldesa_final
ORDER BY alcaldesa_final
"""
pd.read_sql(query, engine)

,alcaldesa_final,avg_contratos,avg_tarjetas_debito,avg_tarjetas_credito,obs
0,0,40426.0,45999.0,10815.0,32574
1,1,53003.0,53671.0,13114.0,9331


In [ ]:
# ── Ejemplo 3: Serie de tiempo — contratos de ahorro por sexo ─────────────────
# Agregamos a nivel nacional por trimestre el número de contratos de ahorro,
# separando mujeres (_m) y hombres (_h).
# Esto permite visualizar la tendencia de la brecha de género en ahorro
# a lo largo de los 17 trimestres (2018Q3–2022Q3).

query = """
SELECT periodo_trimestre,
       SUM(ncont_ahorro_m) AS ahorro_mujeres,
       SUM(ncont_ahorro_h) AS ahorro_hombres
FROM inclusion_financiera
GROUP BY periodo_trimestre
ORDER BY periodo_trimestre
"""
pd.read_sql(query, engine)

,periodo_trimestre,ahorro_mujeres,ahorro_hombres
0,2018Q3,6592.0,6147.0
1,2018Q4,6639.0,6233.0
2,2019Q1,6985.0,6618.0
3,2019Q2,7536.0,7208.0
4,2019Q3,7006.0,6696.0
5,2019Q4,5462.0,5326.0
6,2020Q1,4226.0,4168.0
7,2020Q2,4598.0,4648.0
8,2020Q3,5258.0,5542.0
9,2020Q4,4614.0,4543.0


## 5. Función helper para queries rápidos

In [ ]:
# ── 5. Función helper para queries rápidos ────────────────────────────────────
# Atajo para ejecutar cualquier consulta SQL y obtener un DataFrame.
# Evita repetir pd.read_sql(sql, engine) en cada celda.
# Ejemplo: query("SELECT * FROM inclusion_financiera WHERE estado = 'Oaxaca' LIMIT 10")

def query(sql):
    """Ejecuta una consulta SQL y devuelve un DataFrame."""
    return pd.read_sql(sql, engine)

# Uso:
# query("SELECT * FROM inclusion_financiera WHERE estado = 'Oaxaca' LIMIT 10")

---
### Conexión desde DBeaver
Para conectarte desde DBeaver usa estos parámetros:
- **Host:** `localhost`
- **Port:** `5432`
- **Database:** `tesis_db`
- **Username:** `anapaulaperezgavilan`
- **Password:** *(dejar vacío)*
- **Tabla:** `inclusion_financiera`

---
### Módulo `src/`
Funciones reutilizables extraídas del notebook:
- `src.db` → conexión, carga de tabla, queries
- `src.catalog` → catálogo programático, resumen de NULLs
- `src.plot_style` → estilo centralizado de gráficos

```python
# Ejemplo de uso desde otro script:
from src.db import load_table, query
from src.catalog import build_catalog
from src.plot_style import apply_style
```